In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import collections

import sys
sys.path.append('../')
import plotting

# Read reference data

In [ ]:
ref_df = pd.read_csv('./sequence_data_anonymized_no_duplicates.csv')
ref_df['seq_id_complete'] = ref_df['seq_id']
ref_df['seq_id'] = ref_df['seq_id_anonymized']
ref_df = ref_df.set_index('seq_id')

ref_df

# Read 1D-CNN prediction

In [ ]:
pred_df = pd.read_csv('../data/machine_learning_results/external-validation-predictions.csv')
pred_df.rename(columns={'seq_id_anonymized': 'seq_id'}, inplace=True)
pred_df = pred_df.set_index('seq_id')
pred_df.drop(columns=pred_df.columns.drop('GCfix 2perc'), inplace=True)
pred_df.rename(columns={'GCfix 2perc': 'score'}, inplace=True)

pred_df

# Read experimental data

In [ ]:
pop_df = pd.read_csv(f"../data/internal_datasets/validation_GCall_fix/abundance_by_experiment.csv", dtype={'seq_id': str})
pop_df = pop_df.set_index('seq_id')
pop2_df = pop_df.div(pop_df.mean(axis=0), axis=1)
pop2_df.rename(columns={col: f"x_{col}" for col in pop2_df.columns}, inplace=True)
pop_df = pop_df.join(pop2_df)

pop_df

# Read sequence properties

In [ ]:
prop_df = pd.read_csv(f"../data/internal_datasets/validation_GCall_fix/seqprops.csv", dtype={'id': str})
prop_df.rename(columns={'id': 'seq_id'}, inplace=True)
prop_df = prop_df.set_index('seq_id')

prop_df

# Merge and drop sequences with inserted motif

In [ ]:
df = pd.merge(ref_df, pop_df, left_index=True, right_index=True)
df = pd.merge(df, prop_df, left_index=True, right_index=True)
df = pd.merge(df, pred_df, left_index=True, right_index=True)
df = df.drop(df.loc[df.has_insertedmotif == True].index)

df

# Select sequences which fulfill customary property constraints

In [ ]:
constrained_df = df.loc[(df.GC > 0.4) & (df.GC < 0.6) & (df.hp < 5) & (df.dg > -15)]

constrained_df

# Select same number of sequences based on 1D-CNN score instead

In [ ]:
model_df = df.sort_values('score', ascending=True).head(constrained_df.shape[0])

model_df

# Show the fraction of sequences with less than x% coverage

In [ ]:
data = {'PCR': [], 'n_seqs': [], 'p_seqs': [], 'group': []}
threshold = 0.1
for i in range(1, 6+1):
    for group, idf in zip(['all', 'constrained', '1D-CNN'], [df, constrained_df, model_df]):
        data['PCR'].append(i)
        data['n_seqs'].append(idf.loc[idf[f'x_PCR{i}'] < threshold].shape[0])
        data['p_seqs'].append(idf.loc[idf[f'x_PCR{i}'] < threshold].shape[0] / idf.shape[0])
        data['group'].append(group)

plot_df = pd.DataFrame(data)
plot_df['n_cycles'] = plot_df['PCR'] * 15
plot_df

In [ ]:
fig = px.line(
    plot_df,
    x='n_cycles',
    y='p_seqs',
    color='group',
    color_discrete_map={'all': 'gray', 'constrained': '#3182bd', '1D-CNN': '#de2d26'},
    markers=True,
)

fig.update_yaxes(title="Percent of sequences", range=[0, 0.021], tickformat=".0%", dtick=0.01)
fig.update_xaxes(title="Number of PCR cycles", range=[13, 92], dtick=15, minor_dtick=5)
fig.update_layout(
    width=160,
    height=130,
    margin=dict(l=0, r=5, t=8, b=0),
    showlegend=False,
)

fig = plotting.standardize_plot(fig)
fig.write_image("figure_6_validation_filtering/low_coverage.svg")
fig.show()

# Subsampling sequencing depth

In [ ]:
data = {'depth': [], 'p_seqs': [], 'group': []}
for group, idf in zip(['all', 'constrained', '1D-CNN'], [df, constrained_df, model_df]):
    seqs = [idf.index[k] for k in range(len(idf)) for _ in range(idf['PCR6'].iloc[k])]
    for i in range(1, 50+1):
        for _ in range(30):
            sampled_seqs = collections.Counter(np.random.choice(seqs, size=int(i*len(idf)), replace=False))
            data['depth'].append(i)
            data['p_seqs'].append(len(sampled_seqs)/len(idf))
            data['group'].append(group)

plot_df = pd.DataFrame(data)
plot_df

In [ ]:
# calculate mean and std for every group of 'depth' and 'group'
iplot_df = plot_df.groupby(['depth', 'group']).agg({'p_seqs': ['mean', 'std']}).reset_index()
iplot_df.columns = ['depth', 'group', 'mean', 'std']
iplot_df

In [ ]:
fig = px.line(
    iplot_df,
    y='depth',
    x='mean',
    color='group',
    color_discrete_map={'all': 'gray', 'constrained': '#3182bd', '1D-CNN': '#de2d26'},
)

for threshold in (0.98, 0.99):
    depth_constrained = np.interp(threshold, iplot_df.loc[iplot_df.group == 'constrained', 'mean'], iplot_df.loc[iplot_df.group == 'constrained', 'depth'])
    depth_model = np.interp(threshold, iplot_df.loc[iplot_df.group == '1D-CNN', 'mean'], iplot_df.loc[iplot_df.group == '1D-CNN', 'depth'])
    print(f'Threshold: {threshold}, depth constrained: {depth_constrained}, depth model: {depth_model}')
    
    # add a line between the two points
    fig.add_trace(go.Scatter(
        x=[threshold, threshold],
        y=[depth_constrained, depth_model],
        mode='lines',
        line=dict(color='black', width=1),
        showlegend=False,
    ))



fig.update_xaxes(title="Observed sequences", range=[0.96, 1.0], tickformat=".0%", dtick=0.02, minor_dtick=0.005)
fig.update_yaxes(title="Sequencing depth", range=[0, 50], dtick=25, minor_dtick=5)
fig.update_layout(
    width=160,
    height=130,
    margin=dict(l=0, r=13, t=8, b=0),
    showlegend=False,
)

fig = plotting.standardize_plot(fig)
fig.write_image("figure_6_validation_filtering/sequencing_depth.svg")
fig.show()